# 01 · BLS retail meat prices and CPI

**Milestone 1, items 1.1 and 1.2** · see `docs/MILESTONES.md`

## What we're looking for

We need the monthly U.S. average retail price per pound for beef, pork and chicken
from January 2015 to the latest month available, plus the Consumer Price Index (CPI)
for the same months so we can adjust for inflation later.

BLS publishes several cuts of each meat. In this notebook we pull a few candidates for
each meat, check which ones have complete data for our whole time range, and look at
them side by side. Then we **choose one series per meat** and record why in
`docs/DECISIONS.md`.

**Steps**

1. Load the BLS API key from `.env`
2. List the candidate series
3. Download them from the BLS API
4. Save the raw response, untouched, to `data/raw/`
5. Turn it into a tidy table (one row per series per month)
6. Check coverage: does each series have every month from 2015 to now?
7. Chart the candidates for each meat
8. Choose one series per meat

## Step 1 · Setup and API key

The key lives in the `.env` file at the project root (decision D10), so it never appears
in this notebook or on GitHub. `find_dotenv()` searches upward from the notebook's folder,
so this works even though the notebook sits in `notebooks/`.

We print only whether the key was found and how long it is, never the key itself,
because this notebook will be public.

In [ ]:
import json
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)

import os
BLS_API_KEY = os.getenv("BLS_API_KEY", "").strip()

PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("API key found:", bool(BLS_API_KEY), "| length:", len(BLS_API_KEY))

### How to read it
- **Project root** should end in `meat-prices`.
- **API key found** should be `True`, with a length of about 32 characters.

### If something is wrong
- `False`: the `.env` file is missing, or the line isn't exactly `BLS_API_KEY=yourkey`
  (no spaces, no quotes). Restart the kernel after fixing it.

## Step 2 · Candidate series

Every BLS average-price series has an ID. `APU` means *average price, not seasonally
adjusted*; `0000` means *U.S. city average* (the whole country); the rest identifies the item.

| Meat | Series ID | Item |
|---|---|---|
| Beef | APU0000703112 | Ground beef, 100% beef |
| Pork | APU0000FD3101 | All pork chops |
| Pork | APU0000704211 | Chops, center cut, bone-in |
| Pork | APU0000704212 | Chops, boneless |
| Chicken | APU0000706111 | Chicken, fresh, whole |
| Chicken | APU0000FF1101 | Chicken breast, boneless |
| CPI | CUUR0000SA0 | All items, U.S. city average, not seasonally adjusted |

**Why ground beef for beef?** It's the beef cut most households buy, and it's the price
quoted in most news coverage, so readers can compare our numbers with what they've seen.
We'll still look at the chart before committing.

**Why the not-seasonally-adjusted CPI?** The meat prices aren't seasonally adjusted, so
the CPI we divide by shouldn't be either. Mixing the two would add a seasonal pattern
that isn't really there.

In [ ]:
SERIES = {
    "APU0000703112": {"meat": "Beef",    "item": "Ground beef, 100% beef"},
    "APU0000FD3101": {"meat": "Pork",    "item": "All pork chops"},
    "APU0000704211": {"meat": "Pork",    "item": "Chops, center cut, bone-in"},
    "APU0000704212": {"meat": "Pork",    "item": "Chops, boneless"},
    "APU0000706111": {"meat": "Chicken", "item": "Chicken, fresh, whole"},
    "APU0000FF1101": {"meat": "Chicken", "item": "Chicken breast, boneless"},
    "CUUR0000SA0":   {"meat": "CPI",     "item": "CPI-U, all items (1982-84 = 100)"},
}

START_YEAR = 2015
END_YEAR = date.today().year

pd.DataFrame.from_dict(SERIES, orient="index")

## Step 3 · Download from the BLS API

We use version 2 of the BLS API, which needs the key. With a key, one request can
fetch up to 50 series and 20 years, so all seven series come back in a single call.

The function stops with a clear message if BLS reports a problem, instead of quietly
returning an empty table.

In [ ]:
BLS_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

def fetch_bls(series_ids, start_year, end_year, api_key):
    """Request series from the BLS API v2 and return the parsed JSON response."""
    payload = {
        "seriesid": list(series_ids),
        "startyear": str(start_year),
        "endyear": str(end_year),
        "registrationkey": api_key,
    }
    resp = requests.post(BLS_URL, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") != "REQUEST_SUCCEEDED":
        raise RuntimeError(f"BLS request failed: {data.get('status')} {data.get('message')}")
    return data

raw = fetch_bls(SERIES.keys(), START_YEAR, END_YEAR, BLS_API_KEY)

print("Status:", raw["status"])
for msg in raw.get("message", []):
    print("BLS message:", msg)
print("Series returned:", len(raw["Results"]["series"]))

### How to read it
- **Status** should be `REQUEST_SUCCEEDED`.
- **BLS messages** are warnings. A common one is *"No Data Available for Series X"*,
  which means that series doesn't exist or has no data in our years. That's useful:
  it rules the series out.
- **Series returned** should be 7.

## Step 4 · Save the raw response

We save exactly what BLS sent, before changing anything. If we ever question a number
later, we can go back to the original. The file name includes today's date so
re-downloads don't overwrite earlier pulls. `data/raw/` is not committed to git (D9).

In [ ]:
raw_file = RAW_DIR / f"bls_prices_{date.today():%Y%m%d}.json"
raw_file.write_text(json.dumps(raw, indent=2))
print("Saved:", raw_file.relative_to(PROJECT_ROOT), f"({raw_file.stat().st_size:,} bytes)")

## Step 5 · Tidy table

The API returns nested JSON: a list of series, each with a list of monthly values. We
flatten it into a *tidy* table, one row per series per month, which is the shape pandas
and charting tools work with best.

Details handled here:
- **Periods** come as `M01`…`M12`. `M13` is an annual average on some series; we drop it.
- **Values** come as text. Missing months can appear as `-`, which we turn into a
  proper missing value (`NaN`) rather than an error.
- BLS returns newest first, so we sort oldest first.

In [ ]:
rows = []
for s in raw["Results"]["series"]:
    sid = s["seriesID"]
    for obs in s["data"]:
        if not obs["period"].startswith("M") or obs["period"] == "M13":
            continue
        rows.append({
            "series_id": sid,
            "date": pd.Timestamp(int(obs["year"]), int(obs["period"][1:]), 1),
            "value": obs["value"],
        })

prices = pd.DataFrame(rows)
prices["value"] = pd.to_numeric(prices["value"], errors="coerce")
prices["meat"] = prices["series_id"].map(lambda s: SERIES[s]["meat"])
prices["item"] = prices["series_id"].map(lambda s: SERIES[s]["item"])
prices = prices.sort_values(["series_id", "date"]).reset_index(drop=True)

print(prices.shape)
prices.head(10)

### How to read it
Each row is one month of one series. `value` is dollars per pound for the meats, and
an index number (1982–84 = 100) for the CPI.

## Step 6 · Coverage check

**What we're looking for:** a series we can use needs every month from January 2015
to the latest month, with no gaps. A series that starts late or has holes would leave
blank spots in the app's charts.

In [ ]:
expected_months = pd.date_range(f"{START_YEAR}-01-01", prices["date"].max(), freq="MS")

coverage = (
    prices.dropna(subset=["value"])
    .groupby(["meat", "item", "series_id"])
    .agg(first=("date", "min"), last=("date", "max"), months=("value", "size"))
    .reset_index()
)
coverage["missing_months"] = len(expected_months) - coverage["months"]
coverage["first"] = coverage["first"].dt.strftime("%Y-%m")
coverage["last"] = coverage["last"].dt.strftime("%Y-%m")

missing_from_response = [s for s in SERIES if s not in set(prices["series_id"])]
print("Expected months per series:", len(expected_months))
print("Requested but not returned:", missing_from_response or "none")
coverage

### How to read it
- **first / last:** the earliest and latest months with data.
- **months:** how many months have a value.
- **missing_months:** gaps compared with a complete series. **0 is what we want.**

### What it means
_We'll fill this in together after running: which candidates are complete, and which
are ruled out._

## Step 7 · Chart the candidates

**What we're looking for:** for each meat, do the candidate series tell the same story?
If they rise and fall together, the choice between them matters less, and we can pick
on other grounds, like which one shoppers recognize.

In [ ]:
meats = ["Beef", "Pork", "Chicken"]
fig, axes = plt.subplots(len(meats), 1, figsize=(10, 9), sharex=True)

for ax, meat in zip(axes, meats):
    subset = prices[prices["meat"] == meat]
    for item, g in subset.groupby("item"):
        ax.plot(g["date"], g["value"], label=item, linewidth=1.8)
    ax.set_title(meat, loc="left", fontweight="bold")
    ax.set_ylabel("$ per pound")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=9)

axes[-1].set_xlabel("Month")
fig.suptitle("BLS average retail prices: candidate series", fontsize=13)
fig.tight_layout()
plt.show()

### How to read it
Each panel is one meat; each line is one candidate cut, in dollars per pound at the time
(not yet adjusted for inflation).

### What it means
_We'll fill this in together after running._

## Step 8 · Choose one series per meat

_To decide together after reviewing Steps 6 and 7._ The choice goes into
`docs/DECISIONS.md` as D15, and the chosen series IDs into `docs/DATA_SOURCES.md`.

| Meat | Chosen series | Why |
|---|---|---|
| Beef | | |
| Pork | | |
| Chicken | | |

## What's next and why

**Item 1.3: USDA ERS meat price spreads** (`02_ers_price_spreads.ipynb`). The prices
here show *how much* meat costs. The spreads show *who gets the money*: rancher,
packer or grocery store. That's what we need to answer whether today's prices are
justified.